# Fraud Detection Lab — End-to-End Notebook

This notebook guides you through building a fraud-detection lab environment using a public dataset (Kaggle credit card fraud or Hugging Face datasets). It includes data loading, preprocessing, imbalance handling, model training, evaluation, streaming simulation, and synthetic attack scenarios.

**How to use:**
1. If using Kaggle dataset, place `creditcard.csv` in the working directory or configure Kaggle API credentials.
2. Alternatively, use Hugging Face datasets (an example `liberatoratif/Credit-card-fraud-detection` is referenced).
3. Run cells sequentially and edit parameters where noted.

----
## Sections
1. Setup & Install
2. Load dataset (Kaggle or Hugging Face)
3. Inspect & baseline stats
4. Time-based split
5. Imbalance handling (undersample, SMOTE, anomaly)
6. Baseline model training (RandomForest)
7. Evaluation & metrics
8. Threshold tuning & calibration
9. Streaming simulator & synthetic attack scenarios
10. (Optional) SageMaker pointers

----

_Notebook generated for interactive editing._

## 1) Setup & Install

Run this cell to install required packages. If you're running on local machine, use your environment's package manager instead.

In [ ]:
# Install required packages (uncomment to run)
# Note: If you're on a managed notebook (e.g., SageMaker Studio), some packages may already be installed.
!pip install pandas scikit-learn matplotlib plotly imbalanced-learn kaggle datasets jupyterlab

# If you don't want to install kaggle or datasets, you can manually place 'creditcard.csv' in the notebook folder.
print('Installation step - run as needed in your environment')

## 2) Load dataset (Kaggle or Hugging Face)

You have two options:

### Option A — Kaggle `creditcard.csv` (most common)
1. Create a Kaggle API token and place `kaggle.json` in `~/.kaggle/`.
2. Run the code cell to download the dataset.

### Option B — Hugging Face `datasets` (no API key required for many public datasets)
Use the `datasets` library to load `liberatoratif/Credit-card-fraud-detection` or other available repos.

Choose one option and run the corresponding cell.

In [ ]:
# --- Option A: Download from Kaggle (creditcard.csv) ---
# Uncomment and run this cell after configuring your Kaggle API token (kaggle.json in ~/.kaggle/)

# import os
# os.environ['KAGGLE_CONFIG_DIR'] = os.path.expanduser('~/.kaggle')
# !kaggle datasets download -d mlg-ulb/creditcardfraud -p ./ --unzip

print('Kaggle download: uncomment and run if you have kaggle API configured')

In [ ]:
# --- Option B: Load from Hugging Face datasets ---
# Uncomment to use the datasets library (ensure 'datasets' is installed)
from datasets import load_dataset
ds = load_dataset('liberatoratif/Credit-card-fraud-detection')
df = ds['train'].to_pandas()
df.to_csv('creditcard.csv', index=False)
print('Loaded dataset from Hugging Face and saved to creditcard.csv')

# print('Hugging Face option: uncomment and run if you have datasets library installed')

### If you already have `creditcard.csv` in the working directory, run the cell below to load it.

In [ ]:
import pandas as pd
import os

if os.path.exists('creditcard.csv'):
    df = pd.read_csv('creditcard.csv')
    print('Loaded creditcard.csv with shape:', df.shape)
else:
    print('creditcard.csv not found in current directory. Please download via Kaggle or Hugging Face as shown above.')

## 3) Inspect & Baseline Statistics

Run this cell to inspect data types, class imbalance, and a quick numerical summary.

In [ ]:
# Basic inspection (run after loading df)
try:
    display(df.head())
    print('\nInfo:')
    print(df.info())
    print('\nClass distribution:')
    print(df['Class'].value_counts())
    print('\nDescribe:')
    display(df.describe().T)
except NameError:
    print('DataFrame `df` not found. Load the dataset first.')

## 4) Time-based train/test split

Important: Use time-based split for fraud detection to avoid leakage. The Kaggle dataset has a `Time` column (seconds since first transaction).

In [ ]:
# Time-based split example (adjust split index/time as needed)
try:
    df = df.sort_values('Time').reset_index(drop=True)
    # Choose a split fraction; for example, 70% train / 30% test
    split_frac = 0.7
    split_idx = int(len(df) * split_frac)
    train = df.iloc[:split_idx].copy()
    test = df.iloc[split_idx:].copy()
    print(f'Train shape: {train.shape}, Test shape: {test.shape}')
    print('Train class distribution:')
    print(train['Class'].value_counts(normalize=False))
    print('\nTest class distribution:')
    print(test['Class'].value_counts(normalize=False))
except Exception as e:
    print('Error performing time-based split:', e)

## 5) Imbalance handling

Options: undersample majority, oversample minority (SMOTE), or use anomaly detection techniques. Try each and compare.

In [ ]:
# Example: undersample majority or use SMOTE
from sklearn.model_selection import train_test_split

try:
    X_train = train.drop('Class', axis=1)
    y_train = train['Class']
    X_test = test.drop('Class', axis=1)
    y_test = test['Class']
except NameError:
    raise RuntimeError('Please run the time-based split cell first.')

print('Training sizes (original):', X_train.shape, y_train.value_counts())

# Option A: Undersample majority
def undersample(X, y, ratio=1.0):
    # ratio = number of majority per minority (e.g., 5 means 5 normals per fraud)
    import pandas as pd
    df_concat = pd.concat([X, y], axis=1)
    fraud = df_concat[df_concat['Class'] == 1]
    normal = df_concat[df_concat['Class'] == 0].sample(n=int(len(fraud)*ratio), random_state=42)
    balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)
    Xb = balanced.drop('Class', axis=1)
    yb = balanced['Class']
    return Xb, yb

X_us, y_us = undersample(X_train, y_train, ratio=5)
print('After undersample:', X_us.shape, y_us.value_counts())

# Option B: SMOTE (oversample minority) - uncomment if imbalanced-learn installed
# from imblearn.over_sampling import SMOTE
# sm = SMOTE(random_state=42)
# X_sm, y_sm = sm.fit_resample(X_train, y_train)
# print('After SMOTE:', X_sm.shape, y_sm.value_counts())

print('Choose undersample or SMOTE based on experiments.')

## 6) Baseline model training (Random Forest)

Train a simple RandomForest on the balanced data and evaluate on the (unseen) test set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score

# Use undersampled data by default (adjust if you used SMOTE)
X_train_model, y_train_model = X_us, y_us

clf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
clf.fit(X_train_model, y_train_model)

# Evaluate on test set
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))
print('\nROC-AUC:', roc_auc_score(y_test, y_prob))

# Save model (optional)
import joblib
joblib.dump(clf, 'rf_fraud_baseline.joblib')
print('Saved baseline model to rf_fraud_baseline.joblib')

## 7) Evaluation & Metrics

Plot Precision-Recall curve and inspect thresholds. In fraud detection, PR-AUC is often more informative than ROC-AUC due to class imbalance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
print('Average precision (PR-AUC):', ap)

plt.figure(figsize=(6,4))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve (AP={:.4f})'.format(ap))
plt.grid(True)
plt.show()

## 8) Threshold tuning & calibration

Adjust decision threshold to trade off precision vs recall. Optionally calibrate probabilities using isotonic or sigmoid methods.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Example: tune threshold to target a minimum recall
target_recall = 0.8
best_thresh = 0.5
for t in thresholds:
    preds_t = (y_prob > t).astype(int)
    from sklearn.metrics import recall_score
    if recall_score(y_test, preds_t) >= target_recall:
        best_thresh = t
        break
print('Threshold achieving at least recall', target_recall, '=>', best_thresh)

# Optional: probability calibration (uncomment to run)
# calibrator = CalibratedClassifierCV(clf, method='isotonic', cv='prefit')
# calibrator.fit(X_train_model, y_train_model)
# y_prob_cal = calibrator.predict_proba(X_test)[:,1]

## 9) Streaming simulator & synthetic attack scenarios

Turn the test set into a streaming generator and inject synthetic fraud patterns to simulate attacks.

In [ ]:
import numpy as np
import time
from sklearn.metrics import precision_score, recall_score

def transaction_stream(df, delay=0.0):
    # Yields rows as pandas Series (do not sleep if delay=0)
    for _, row in df.iterrows():
        yield row

# Example attack scenarios
def add_micro_fraud(df, n=200, amount_col='Amount'):
    sample = df.sample(n, replace=True).copy()
    sample[amount_col] = np.random.uniform(0.1, 3.0, size=len(sample))
    sample['Class'] = 1
    return pd.concat([df, sample]).reset_index(drop=True)

def add_high_value_fraud(df, n=50, amount_col='Amount'):
    sample = df.sample(n, replace=True).copy()
    sample[amount_col] = np.random.uniform(5000, 15000, size=len(sample))
    sample['Class'] = 1
    return pd.concat([df, sample]).reset_index(drop=True)

def burst_attack(df, center_time, size=30, time_col='Time'):
    sample = df.sample(size, replace=True).copy()
    sample[time_col] = center_time + np.random.randint(-30, 30, size=size)
    sample['Class'] = 1
    return pd.concat([df, sample]).reset_index(drop=True)

print('Defined simulator and attack scenario functions')

In [ ]:
# Example: create a simulated stream with injected attack and evaluate online detection
try:
    test_stream_df = test.copy().reset_index(drop=True)
    # Inject micro frauds
    test_aug = add_micro_fraud(test_stream_df, n=200)
    # Sort by time if available
    if 'Time' in test_aug.columns:
        test_aug = test_aug.sort_values('Time').reset_index(drop=True)

    # Simulate streaming and online scoring (fast loop, no sleep)
    y_true_stream = []
    y_pred_stream = []
    for _, row in test_aug.iterrows():
        x = row.drop('Class').values.reshape(1,-1)
        prob = clf.predict_proba(x)[:,1][0]
        pred = int(prob > 0.5)  # default threshold
        y_true_stream.append(row['Class'])
        y_pred_stream.append(pred)

    from sklearn.metrics import classification_report
    print('Stream Classification Report:')
    print(classification_report(y_true_stream, y_pred_stream))
except Exception as e:
    print('Error running streaming example:', e)

## 10) Optional: SageMaker Pipeline Pointers

If you want to run this pipeline on SageMaker, consider:

- Uploading `creditcard.csv` to S3 and using SageMaker Processing for feature engineering.
- Training with SageMaker Training Jobs (XGBoost built-in or a custom container).
- Using Model Registry + Pipelines for CI/CD.
- Deploying as Real-Time Endpoint or Batch Transform for production scoring.

I can generate a SageMaker-specific script or CloudFormation template on request.

----

# Save & Download

The notebook file has been saved as **/mnt/data/fraud_detection_lab.ipynb**. Download it, open in JupyterLab or VS Code, and run cells interactively. Edit scenario parameters to create different attack types and experiment with different models (XGBoost, LightGBM, Autoencoders).

If you'd like, I can also: 
- Create an Anki deck from the flashcards
- Produce a SageMaker-ready version of this notebook
- Expand attack scenarios and monitoring (CloudWatch alerts, Lambda actions)
